In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt

# -----------------------------
# Load FIRMS shapefile
# -----------------------------
firms = gpd.read_file("SHP_data/fire_nrt_SV-C2_700748.shp")
print("FIRMS columns:", firms.columns)

# If CRS is missing, set it (FIRMS is typically WGS84 lat/lon)
if firms.crs is None:
    firms = firms.set_crs("EPSG:4326")

print("FIRMS CRS:", firms.crs)

# Date filter: Nov 3 2025 .. Nov 10 2025 (inclusive)
firms["ACQ_DATE"] = pd.to_datetime(firms["ACQ_DATE"], errors="coerce")
start_date = pd.Timestamp("2025-11-03")
end_date   = pd.Timestamp("2025-11-10")

firms_t = firms[(firms["ACQ_DATE"] >= start_date) & (firms["ACQ_DATE"] <= end_date)].copy()
print("FIRMS points in range:", len(firms_t))
print("Date min/max:", firms_t["ACQ_DATE"].min(), firms_t["ACQ_DATE"].max())

# Buffer radius (meters)
buffer_m = 750

# -----------------------------
# Gather TIFF files
# -----------------------------
folder = "try-1"
tiffs = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(folder)
    for f in files if f.endswith(".tiff")
])

print(f"Total TIFF files found: {len(tiffs)}")

# -----------------------------
# Helper normalization
# -----------------------------
def norm(x):
    x = x - np.nanmin(x)
    xmax = np.nanmax(x)
    return x / xmax if xmax > 0 else x

# -----------------------------
# Loop over first 10 images
# -----------------------------
for i, path in enumerate(tiffs[:10]):

    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
        raster_crs = src.crs
        transform = src.transform
        H, W = src.height, src.width

    # Sentinel Hub evalscript band order:
    # 0:B02, 1:B03, 2:B04, 3:B08, 4:B12
    B02, B03, B04, B08, B12 = img[0], img[1], img[2], img[3], img[4]

    rgb_image = np.dstack([norm(B04), norm(B03), norm(B02)])
    false_color = np.dstack([norm(B12), norm(B08), norm(B04)])
    nbr = (B08 - B12) / (B08 + B12 + 1e-8)

    # -----------------------------
    # FIRMS -> raster GT for THIS TIFF (use its CRS/transform/H/W)
    # -----------------------------
    firms_local = firms_t

    # Reproject FIRMS once per raster CRS (safe)
    if firms_local.crs != raster_crs:
        firms_local = firms_local.to_crs(raster_crs)

    # IMPORTANT: buffer on projected CRS in meters
    # If CRS is geographic (degrees), buffer would be wrong.
    # Many GeoTIFFs are projected; if not, you must project first.
    if raster_crs.is_geographic:
        # quick fix: project FIRMS to a metric CRS for buffering then back
        # (uses Web Mercator as a generic metric CRS)
        firms_metric = firms_local.to_crs("EPSG:3857")
        firms_metric["geometry"] = firms_metric.geometry.buffer(buffer_m)
        firms_buff = firms_metric.to_crs(raster_crs)
    else:
        firms_buff = firms_local.copy()
        firms_buff["geometry"] = firms_buff.geometry.buffer(buffer_m)

    firms_shapes = [(geom, 1) for geom in firms_buff.geometry if geom is not None]

    firms_mask = rasterize(
        firms_shapes,
        out_shape=(H, W),
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    gt_firms = np.zeros((H, W), dtype=np.uint8)
    gt_firms[firms_mask == 1] = 2

    # -----------------------------
    # Plot: RGB, False color, NBR, FIRMS GT overlay
    # -----------------------------
    plt.figure(figsize=(24, 6))
    plt.suptitle(f"Image {i+1}: {os.path.basename(path)}", fontsize=16)

    plt.subplot(1, 4, 1)
    plt.title("1. True RGB (Natural)")
    plt.imshow(rgb_image)
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.title("2. False Color (SWIR-NIR-RED)")
    plt.imshow(false_color)
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.title("3. NBR Index")
    im = plt.imshow(nbr, cmap="RdYlGn")
    plt.colorbar(im, fraction=0.046, pad=0.04, label="NBR")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.title(f"4. FIRMS GT (buffer={buffer_m}m)")
    plt.imshow(false_color)
    overlay = np.zeros((H, W, 4), dtype=np.float32)
    overlay[..., 1] = 1.0  # green
    overlay[..., 3] = (gt_firms == 2).astype(np.float32) * 0.65
    plt.imshow(overlay)
    plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def show_mask_overlay(base_rgb01, gt_label, title="", alpha=0.6, draw_border=True):
    """
    base_rgb01: (H,W,3) float in [0,1] (e.g., false_color)
    gt_label: (H,W) uint8 label mask where fire pixels are 2
    """
    H, W = base_rgb01.shape[:2]
    assert gt_label.shape == (H, W), f"Shape mismatch: base={base_rgb01.shape}, gt={gt_label.shape}"

    m = (gt_label == 2)

    plt.imshow(base_rgb01)
    overlay = np.zeros((H, W, 4), dtype=np.float32)
    overlay[..., 1] = 1.0  # green
    overlay[..., 3] = m.astype(np.float32) * alpha
    plt.imshow(overlay)

    if draw_border:
        m_u8 = (m.astype(np.uint8) * 255)
        contours, _ = cv2.findContours(m_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        border = np.zeros((H, W, 4), dtype=np.float32)
        for c in contours:
            cv2.drawContours(border, [c], -1, (1.0, 1.0, 1.0, 1.0), thickness=2)
        plt.imshow(border)

    plt.title(title)
    plt.axis("off")


# Example: visualize FIRMS GT
gt = gt_firms  # or gt_sam_indices

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title("GT mask (binary)  (label==2)")
plt.imshow((gt == 2).astype(np.uint8), cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
show_mask_overlay(false_color, gt, title="GT overlay on false color", alpha=0.65, draw_border=True)

plt.tight_layout()
plt.show()
